# Stage 5 — Fine-Tune YOLOv8 on Football Data (Ball, Player, Referee)

Run this notebook in **Google Colab** with a GPU runtime:
`Runtime > Change runtime type > T4 GPU`

This fine-tunes pretrained `yolov8n.pt` (not training from scratch) on the
Roboflow Detect-Players dataset (1,462 images, 3 classes: Ball, Player, Referee).

In [1]:
# Mount Google Drive - this is WHY we backed up data there earlier.
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install ultralytics (YOLOv8) in this Colab session
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 9.0 MB/s eta 0:00:00


## Upload the dataset

Two options:
1. **Upload the `roboflow_players` folder to your Google Drive first**, then point to it below (recommended - survives Colab session restarts)
2. Upload directly to this Colab session using the file browser on the left (faster, but lost when the session ends)

Adjust `DATASET_PATH` below to match wherever you placed it.

In [7]:
import shutil
from pathlib import Path

# EDIT THIS to match your actual Drive path once uploaded
DATASET_PATH = Path('/content/drive/MyDrive/dataset/sport_vision/backup_staging/roboflow_players/Detect Players.v7i.yolov8')

assert DATASET_PATH.exists(), f"Dataset not found at {DATASET_PATH} - check your Drive upload path."
print(f"Dataset found: {DATASET_PATH}")
print(f"Contents: {list(DATASET_PATH.iterdir())}")

Dataset found: /content/drive/MyDrive/dataset/sport_vision/backup_staging/roboflow_players/Detect Players.v7i.yolov8
Contents: [PosixPath('/content/drive/MyDrive/dataset/sport_vision/backup_staging/roboflow_players/Detect Players.v7i.yolov8/train'), PosixPath('/content/drive/MyDrive/dataset/sport_vision/backup_staging/roboflow_players/Detect Players.v7i.yolov8/test'), PosixPath('/content/drive/MyDrive/dataset/sport_vision/backup_staging/roboflow_players/Detect Players.v7i.yolov8/data.yaml'), PosixPath('/content/drive/MyDrive/dataset/sport_vision/backup_staging/roboflow_players/Detect Players.v7i.yolov8/README.dataset.txt'), PosixPath('/content/drive/MyDrive/dataset/sport_vision/backup_staging/roboflow_players/Detect Players.v7i.yolov8/README.roboflow.txt'), PosixPath('/content/drive/MyDrive/dataset/sport_vision/backup_staging/roboflow_players/Detect Players.v7i.yolov8/valid')]


## Fix data.yaml paths

The downloaded `data.yaml` uses relative paths (`../train/images`) that assumed
a specific local folder structure. In Colab, we rewrite it with absolute paths
pointing at wherever YOUR dataset actually sits - this avoids a very common
'file not found' error when the dataset is moved between environments.

In [8]:
import yaml

data_yaml_path = DATASET_PATH / 'data.yaml'
with open(data_yaml_path) as f:
    data_config = yaml.safe_load(f)

data_config['train'] = str(DATASET_PATH / 'train' / 'images')
data_config['val'] = str(DATASET_PATH / 'valid' / 'images')
data_config['test'] = str(DATASET_PATH / 'test' / 'images')

fixed_yaml_path = '/content/data_fixed.yaml'
with open(fixed_yaml_path, 'w') as f:
    yaml.dump(data_config, f)

print(f"Classes: {data_config['names']}")
print(f"Fixed config saved to: {fixed_yaml_path}")

Classes: ['Ball', 'Player', 'Referee']
Fixed config saved to: /content/data_fixed.yaml


## Fine-tune YOLOv8

Starting from the pretrained `yolov8n.pt` (COCO weights) - NOT training from
scratch. `epochs=50` is a reasonable starting point for ~1,500 images; adjust
based on how training progresses (watch the loss curves in the output).

In [9]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # pretrained starting point

results = model.train(
    data=fixed_yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    project='/content/drive/MyDrive/sports-vision-backup/yolo_runs',
    name='football_finetune_v1',
    patience=10,  # stop early if validation loss plateaus for 10 epochs
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.171 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data_fixed.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freez

## Evaluate: Precision, Recall, mAP

This directly answers your Stage 5 'Measure' checklist item.

In [10]:
metrics = model.val()

print(f"\n=== EVALUATION RESULTS ===")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

# Per-class breakdown - important for checking ball detection specifically,
# since that was flagged as the weak point during local pretrained testing
print(f"\n=== PER-CLASS mAP50 ===")
for i, class_name in data_config['names'].items() if isinstance(data_config['names'], dict) else enumerate(data_config['names']):
    print(f"  {class_name}: {metrics.box.ap50[i]:.4f}")

Ultralytics 8.4.171 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.7±0.2 ms, read: 71.5±44.5 MB/s, size: 115.1 KB)
val: Scanning /content/drive/MyDrive/dataset/sport_vision/backup_staging/roboflow_players/Detect Players.v7i.yolov8/valid/labels.cache... 274 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 274/274 76.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 1.9it/s 9.5s
                   all        274       4189      0.865      0.768      0.796      0.503
                  Ball        202        202      0.774       0.49        0.5      0.226
                Player        274       3647       0.94      0.972      0.983      0.672
               Referee        210        340       0.88      0.841      0.904      0.609
Speed: 2.0ms preprocess, 4.3ms inference, 0.0ms loss, 

## Save fine-tuned weights back to Drive

The trained weights are small (a few MB) - this is the only thing you need
to bring back to your LOCAL machine, not the full dataset or training logs.

In [12]:
import shutil

best_weights = '/content/drive/MyDrive/dataset/sport_vision/backup_staging/yolo_runs/football_finetune_v1/weights/best.pt'
print(f"Fine-tuned weights saved at: {best_weights}")
print("Download this single file from Drive to your local project's models/ folder.")

Fine-tuned weights saved at: /content/drive/MyDrive/dataset/sport_vision/backup_staging/yolo_runs/football_finetune_v1/weights/best.pt
Download this single file from Drive to your local project's models/ folder.


In [13]:
import os
weights_path = '/content/drive/MyDrive/sports-vision-backup/yolo_runs/football_finetune_v1/weights/best.pt'
print(f"Exists: {os.path.exists(weights_path)}")

# If that's False, search for it instead
import subprocess
result = subprocess.run(['find', '/content/drive/MyDrive', '-name', 'best.pt'], capture_output=True, text=True)
print("Found at:", result.stdout)

Exists: True
Found at: /content/drive/MyDrive/sports-vision-backup/yolo_runs/football_finetune_v1/weights/best.pt

